# 03 · Experiments with known ground truth
On real data we never observe an individual's treatment effect. Two experiments get around this:
1. **Semi-synthetic benchmark:** real Criteo covariates and a realistic baseline, with *simulated* outcomes whose individual
   effect is **known**. Learners can then be scored on PEHE (RMSE of the individual effect), not only on ranking.
2. **Observational bias:** we turn the RCT into *confounded* observational data, where users likely to visit anyway are more
   likely to be treated, and test which estimators recover the true (randomised) average effect.

**Runtime: CPU**

In [ ]:
# ---------- Setup: run this cell first (CPU runtime is enough) ----------
import os, sys, subprocess, json, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
SMOKE = os.environ.get("UP_SMOKE") == "1"          # local smoke test on synthetic data
try:
    import google.colab  
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
if SMOKE:
    PROJECT_DIR = os.environ["UP_PROJECT_DIR"]
elif ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    import glob
    hits = sorted(glob.glob("/content/drive/MyDrive/**/uplift/src/uplift/data.py", recursive=True), key=len)
    assert hits, "Upload the 'uplift' folder to your Google Drive first."
    PROJECT_DIR = str(Path(hits[0]).parents[2])                     # found automatically
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabulate"], check=True)
else:   # local Jupyter / VS Code: notebook lives in uplift/notebooks/
    PROJECT_DIR = str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.insert(0, f"{PROJECT_DIR}/src")
ART, FIG = f"{PROJECT_DIR}/artifacts", f"{PROJECT_DIR}/reports/figures"
DATA = "/content/data" if ON_COLAB else f"{PROJECT_DIR}/data"
for d in (ART, FIG, DATA): os.makedirs(d, exist_ok=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110
from uplift.data import FEATURES, load_criteo, make_synthetic
from uplift import experiment as E, learners as L, metrics as M, causal as C
SEED = 42
df = make_synthetic(60_000) if SMOKE else load_criteo(DATA, sample_frac=0.10)   # 10% sample ~ 1.4M users
TREAT_SHARE = 0.85                                                              # by design of the experiment
def savefig(name):
    plt.tight_layout(); plt.savefig(f"{FIG}/{name}.png", dpi=160, bbox_inches="tight"); plt.show()
def save_json(o, name):
    json.dump(o, open(f"{ART}/{name}.json", "w"), indent=2, default=float)
def load_json(name):
    return json.load(open(f"{ART}/{name}.json"))
print(f"{len(df):,} users | treated share {df.treatment.mean():.3f} | visit {df.visit.mean():.4f} | conversion {df.conversion.mean():.5f}")

## 1. Semi-synthetic benchmark

In [ ]:
N_SS = 20_000 if SMOKE else 300_000
# A realistic baseline-visit model, fit on a SEPARATE 20% of users. Using the same users later would leak their
# own outcomes into the simulated treatment assignment (in-sample fit memorises noise) and break unconfoundedness.
fit_part = np.random.default_rng(SEED).random(len(df)) < 0.2
dfA, dfB = df[fit_part], df[~fit_part].reset_index(drop=True)
ctrlA = dfA[dfA.treatment == 0]
base_fn = C.baseline_logit_model(ctrlA[FEATURES].values, ctrlA.visit.values, seed=SEED)
ss = dfB.sample(min(N_SS, len(dfB)), random_state=SEED).reset_index(drop=True)
X = ss[FEATURES].values
base = base_fn(X)
SS_LEARNERS = [L.SLearner, L.TLearner, L.XLearner, L.DRLearner, L.TransformedOutcome, L.ResponseModel]
rows = []
for scenario in ["heterogeneous", "weak"]:
    for rep in range(1 if SMOKE else 3):
        t, y, tau = C.semi_synthetic(X, base, scenario, TREAT_SHARE, seed=rep)
        h = len(y) // 2                       # train on first half, evaluate on second
        for Cls in SS_LEARNERS:
            kw = {"propensity": TREAT_SHARE} if Cls in (L.XLearner, L.DRLearner, L.TransformedOutcome) else {}
            m = Cls(seed=rep, **kw).fit(X[:h], t[:h], y[:h]); th = m.predict(X[h:])
            rows.append({"scenario": scenario, "rep": rep, "model": m.name,
                         "pehe": C.pehe(th, tau[h:]) if "Response" not in m.name else np.nan,
                         "rank_corr_with_true_tau": pd.Series(th).corr(pd.Series(tau[h:]), method="spearman"),
                         "qini_coef": M.qini_coefficient(y[h:], t[h:], th),
                         "ate_error": (th.mean() - tau[h:].mean()) if "Response" not in m.name else np.nan})
        print(scenario, rep, "done")
bench = pd.DataFrame(rows)
bench_sum = bench.groupby(["scenario", "model"])[["pehe", "rank_corr_with_true_tau", "qini_coef", "ate_error"]].agg(["mean", "std"])
bench.to_csv(f"{ART}/semi_synthetic_runs.csv", index=False); bench_sum.round(4)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
d = bench[bench.scenario == "heterogeneous"]
sns.barplot(data=d[~d.model.str.contains("Response")], x="pehe", y="model", ax=ax[0], color="#4C72B0", errorbar="sd")
ax[0].set_title("PEHE: error in individual effects (lower is better)")
sns.barplot(data=d, x="rank_corr_with_true_tau", y="model", ax=ax[1], color="#55A868", errorbar="sd")
ax[1].set_title("Spearman correlation with the TRUE effect"); ax[1].set_ylabel("")
savefig("03_semi_synthetic")

In [ ]:
# Does the Qini coefficient (computable on real data) agree with the true-effect metrics (not computable)?
agree = bench.groupby(["scenario", "model"])[["qini_coef", "rank_corr_with_true_tau"]].mean()
print("Spearman(Qini, true rank-corr) across models:",
      round(agree.groupby(level=0).apply(lambda g: g.qini_coef.corr(g.rank_corr_with_true_tau, method="spearman")).mean(), 3))
agree.round(4)

## 2. Observational bias
We sub-sample the RCT so that **users with a high baseline visit probability are more likely to be treated**, which is
typical of real campaigns that target "good" users. The sub-sampling is designed so that the covariate distribution is
unchanged, so the true ATE equals the randomised estimate on the same users. The model that defines "baseline visit
probability" is fit on a separate 20% of users, to avoid leaking outcomes into treatment assignment.

In [ ]:
Xall = dfB[FEATURES].values                       # users NOT used to fit base_fn
z = base_fn(Xall); z = (z - z.mean()) / z.std()
truth = E.ate_binary(dfB.visit, dfB.treatment)
print(f"Randomised (truth) ATE = {truth['ate']:.5f}  [{truth['ci_low']:.5f}, {truth['ci_high']:.5f}]")
obs_rows = []
for strength in [0.5, 1.0, 1.5]:
    s = np.clip(1 / (1 + np.exp(-strength * z)), 0.05, 0.95)       # true propensity in the confounded sample
    keep = C.confounded_subsample(dfB.treatment.values, s, TREAT_SHARE, seed=SEED)
    est = C.ate_estimators(Xall[keep], dfB.treatment.values[keep], dfB.visit.values[keep], oracle_propensity=s[keep], seed=SEED)
    est["confounding_strength"], est["n"] = strength, int(keep.sum())
    est["bias"] = est.ate - truth["ate"]; est["covers_truth"] = (est.ci_low <= truth["ate"]) & (truth["ate"] <= est.ci_high)
    obs_rows.append(est.reset_index())
    print(f"strength={strength}: n={keep.sum():,}  naive bias={est.loc['Naive difference in means','bias']:+.5f}  "
          f"AIPW bias={est.loc['AIPW / doubly robust','bias']:+.5f}")
obs = pd.concat(obs_rows); obs.to_csv(f"{ART}/observational_bias.csv", index=False)
obs.pivot(index="estimator", columns="confounding_strength", values="bias").round(5)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for i, (name, g) in enumerate(obs.groupby("estimator", sort=False)):
    ax.errorbar(g.confounding_strength + (i - 2) * 0.05, g.ate, yerr=[g.ate - g.ci_low, g.ci_high - g.ate], fmt="o", capsize=3, label=name)
ax.axhline(truth["ate"], c="k", ls="--", label="randomised truth"); ax.axhspan(truth["ci_low"], truth["ci_high"], color="grey", alpha=.2)
ax.set_xlabel("confounding strength"); ax.set_ylabel("estimated ATE (visit)"); ax.legend(fontsize=8)
ax.set_title("Recovering the experimental effect from confounded data"); savefig("03_observational_bias")

## 3. Auto-generated results summary (uses notebooks 01–03)

In [ ]:
ex, mo = load_json("experiment"), load_json("modeling")
comp = pd.read_csv(f"{ART}/model_comparison_visit.csv", index_col=0)
conv = pd.read_csv(f"{ART}/model_comparison_conversion.csv", index_col=0)
pol = pd.read_csv(f"{ART}/policy.csv", index_col=0)
v, cv = ex["ate"]["visit"], ex["ate"]["conversion"]
lines = ["# Results summary (auto-generated by notebook 03)", "",
 f"* Data: {ex['n']:,} users (10% stratified sample of Criteo Uplift v2.1); treated share {ex['srm']['share_treated']:.4f} "
 f"(SRM p = {ex['srm']['p_value']:.3f}); max |SMD| = {ex['max_abs_smd']:.4f}",
 f"* **Visit ATE: {v['rate_control']:.4%} → {v['rate_treated']:.4%} = +{v['ate']*100:.3f} pp "
 f"[{v['ci_low']*100:.3f}, {v['ci_high']*100:.3f}], relative +{v['relative_lift']:.1%}, p = {v['p_value']:.1e}**",
 f"* Conversion ATE: +{cv['ate']*100:.4f} pp [{cv['ci_low']*100:.4f}, {cv['ci_high']*100:.4f}], relative +{cv['relative_lift']:.1%}",
 f"* Best uplift model (visit): **{mo['best_model']}**, Qini coefficient {comp.loc[mo['best_model'],'qini_coef']:.4f} "
 f"[{comp.loc[mo['best_model'],'qini_ci_low']:.4f}, {comp.loc[mo['best_model'],'qini_ci_high']:.4f}] vs response model "
 f"{comp.loc['Response model (baseline)','qini_coef']:.4f}",
 f"* Uplift in top 10% (best model): {comp.loc[mo['best_model'],'uplift@10%']*100:.3f} pp vs average {v['ate']*100:.3f} pp",
 "", "## Experiment", pd.DataFrame(ex["ate"]).T[["rate_treated","rate_control","ate","ci_low","ci_high","relative_lift","p_value"]].to_markdown(),
 "", "## Power", pd.DataFrame(ex["power"]).to_markdown(index=False),
 "", "## Uplift models: visit (holdout)", comp.round(4).to_markdown(),
 "", "## Uplift models: conversion (holdout)", conv.round(4).to_markdown(),
 "", f"## Targeting policy (value/visit={mo['value']}, cost/user={mo['cost']})", pol.round(3).to_markdown(),
 "", "## Semi-synthetic benchmark (mean ± sd over reps)", bench_sum.round(4).to_markdown(),
 "", "## Observational bias", obs.round(5).to_markdown(index=False)]
open(f"{PROJECT_DIR}/reports/results_summary.md", "w").write("\n".join(lines)); print("\n".join(lines[:8]))